In [1]:
import numpy as np
import pandas as pd
import os
import textwrap

In [2]:
def load_data(data_dir, x_filename='x_train.csv', y_filename='y_train.csv'):
    x_train_df = pd.read_csv(os.path.join(data_dir, x_filename))
    y_train_df = pd.read_csv(os.path.join(data_dir, y_filename))

    N, n_cols = x_train_df.shape
    print("Shape of x_train_df: (%d, %d)" % (N, n_cols))
    print("Shape of y_train_df: %s" % str(y_train_df.shape))

    # Print out 8 random entries
    tr_text_list = x_train_df['text'].values.tolist()
    prng = np.random.RandomState(101)
    rows = prng.permutation(np.arange(y_train_df.shape[0]))
    for row_id in rows[:8]:
        text = tr_text_list[row_id]
        print("row %5d | %s BY %s | y = %s" % (
            row_id,
            y_train_df['title'].values[row_id],
            y_train_df['author'].values[row_id],
            y_train_df['Coarse Label'].values[row_id],
            ))
        # Pretty print text via textwrap library
        line_list = textwrap.wrap(tr_text_list[row_id],
            width=70,
            initial_indent='  ',
            subsequent_indent='  ')
        print('\n'.join(line_list))
        print("")
    return x_train_df, y_train_df

In [3]:
x_train_df, y_train_df = load_data(data_dir = "data", x_filename = "x_train.csv", y_filename = "y_train.csv")

Shape of x_train_df: (5557, 32)
Shape of y_train_df: (5557, 5)
row  4746 | The Red and the Black: A Chronicle of 1830 BY Stendhal | y = Key Stage 4-5
  It was hermetically sealed; he was on the point of  fainting and
  remained for a long time leaning against the oak; then  with a
  staggering step he went to have another look at the gardener's
  ladder. The chain which he had once forced asunder--in, alas, such
  different  circumstances--had not yet been repaired. Carried away by
  a moment of  madness, Julien pressed it to his lips.

row  1250 | Cranford BY Elizabeth Cleghorn Gaskell | y = Key Stage 4-5
  Miss Pole, Miss Matty, and I, meanwhile attended to Miss Brown: and
  hard  work we found it to relieve her querulous and never-ending
  complaints. But if we were so weary and dispirited, what must Miss
  Jessie have been! Yet she came back almost calm as if she had gained
  a new strength. She  put off her mourning dress, and came in,
  looking pale and gentle,  thanking us each 

In [4]:
def tokenize_text(raw_text):
    ''' Transform a plain-text string into a list of tokens
    
    We assume that *whitespace* divides tokens.
    
    Args
    ----
    raw_text : string
    
    Returns
    -------
    list_of_tokens : list of strings
        Each element is one token in the provided text
    '''
    list_of_tokens = raw_text.split() # split method divides on whitespace by default
    for pp in range(len(list_of_tokens)):
        cur_token = list_of_tokens[pp]
        # Remove punctuation
        for punc in ['?', '!', '_', '.', ',', '"', '/']:
            cur_token = cur_token.replace(punc, "")
        # Turn to lower case
        clean_token = cur_token.lower()
        # Replace the cleaned token into the original list
        list_of_tokens[pp] = clean_token
    return list_of_tokens

In [5]:
training_text = x_train_df['text'].values.tolist()
tokenized_training_text = [tokenize_text(text) for text in training_text]
print("Example tokenized texts:")
for i, tokens in enumerate(tokenized_training_text[:5]):
    print(f"Text {i}: {tokens}")

Example tokenized texts:
Text 0: ['yes', 'what', 'sort', 'of', 'terms', 'was', 'he', 'on', 'with', 'the', 'guests—you', 'and', 'miss', 'norris', 'and', 'all', 'of', 'them', 'just', 'polite', 'and', 'rather', 'silent', 'you', 'know', 'keeping', 'himself', 'to', 'himself', 'we', "didn't", 'see', 'so', 'very', 'much', 'of', 'him', 'except', 'at', 'meals', 'we', 'were', 'here', 'to', 'enjoy', 'ourselves', 'and—well', 'he', "wasn't", 'he', "wasn't", 'there', 'when', 'the', 'ghost', 'walked', 'no', 'i', 'heard', 'mark', 'calling', 'for', 'him', 'when', 'he', 'went', 'back', 'to', 'the', 'house', 'i', 'expect', 'cayley', 'stroked', 'down', 'his', 'feathers', 'a', 'bit', 'and', 'told', 'him', 'that', 'girls', 'will', 'be', 'girls—hallo', 'here', 'we', 'are']
Text 1: ['perhaps', 'i', 'should', 'say', 'that', 'it', 'was', "mark's", 'private', 'plan', 'my', 'own', 'was', 'different', 'the', 'announcement', 'at', 'breakfast', 'went', 'well', 'after', 'the', 'golfing-party', 'had', 'gone', 'off', '

In [6]:
def build_vocabulary(tokenized_text_list):
  tok_count_dict = dict()

  for tokenized_text in tokenized_text_list:
    for token in tokenized_text:
      if token not in tok_count_dict:
        tok_count_dict[token] = 1 # Initialize count for new token
      else:
        tok_count_dict[token] += 1 # Increment count for existing token
  
  # Sort the tokens by their counts in descending order
  sorted_tokens = list(sorted(tok_count_dict, key=tok_count_dict.get, reverse=True))

  # Print out the 10 most common tokens and their counts
  for w in sorted_tokens[:10]:
    print("%5d %s" % (tok_count_dict[w], w))
  
  # Print out the 10 least common tokens and their counts
  for w in sorted_tokens[-10:]:
    print("%5d %s" % (tok_count_dict[w], w))

  vocab_list = [w for w in sorted_tokens if tok_count_dict[w] >= 4]
  
  return vocab_list

In [7]:
vocab_list = build_vocabulary(tokenized_training_text)
print(f"Vocabulary size: {len(vocab_list)}")
print("Example vocabulary tokens:", vocab_list[:10])

24173 the
14483 and
12029 of
11903 to
 9227 a
 6778 in
 6728 i
 5982 he
 5501 that
 5258 was
    1 honorable;
    1 saucy
    1 brandied
    1 absinthe
    1 teased
    1 coquenard
    1 madinier
    1 hearse
    1 lumpy
    1 monumental
Vocabulary size: 8116
Example vocabulary tokens: ['the', 'and', 'of', 'to', 'a', 'in', 'i', 'he', 'that', 'was']
